# 🚀 UD-03-600 Week03 강의 연동 실습

Week03 전처리 & 형태소 분석 — split vs morphs, clean_title(), Pecab/Okt 비교, 전처리 계약

> **🖥️ 환경 설정**: Google Colab에서 실행하세요.
> 추가 패키지 필요 시: `!pip install pecab konlpy -q`

> **⌨️ 단축키 안내**
> | 단축키 | 동작 |
> |--------|------|
> |  | 셀 실행 후 다음 셀 이동 |
> |  | 위에 셀 삽입 |
> |  | 아래에 셀 삽입 |
> |  | 셀 삭제 |

In [1]:
# 📌 §Setup 환경 설정
# [Colab 전용] pecab 설치 — 순수 Python, Java 불필요
# !pip install pecab konlpy -q

# 설치 확인
from pecab import PeCab
pecab = PeCab()
print("Pecab 준비 완료:", pecab.morphs("안녕하세요"))
# 예상 출력: Pecab 준비 완료: ['안녕', '하', '세요']

Pecab 준비 완료: ['안녕', '하', '세요']


## 🎯 P1 — split vs morphs 관찰

`split()`과 `PeCab.morphs()`의 차이를 3문장으로 직접 확인한다.  
실행 전에 결과를 **예측**해보세요.


🔗 [PeCab GitHub](https://github.com/hyunwoongko/pecab)

[→ §100 강의노트](../notes/UD-03-100__split-to-morphs.md)

In [2]:
# 📌 §100 핵심비교 — split vs morphs
sentences = [
    "아버지가방에들어가시다",
    "나는학교에간다",
    "차를마시며차를탔다",
]
# ✍️ 각 문장의 split()과 morphs() 결과를 for 루프로 출력해보세요
# 코드를 작성해주세요 👇
for s in sentences:
    print(f"입력: {s}")
    print(f"  split  : {s.split()}")
    print(f"  morphs : {pecab.morphs(s)}")
    print()

# 예상 출력:
# 입력: 아버지가방에들어가시다
#   split  : ['아버지가방에들어가시다']
#   morphs : ['아버지', '가', '방', '에', '들어가', '시', '다']

입력: 아버지가방에들어가시다
  split  : ['아버지가방에들어가시다']
  morphs : ['아버지', '가', '방', '에', '들어가', '시', '다']

입력: 나는학교에간다
  split  : ['나는학교에간다']
  morphs : ['나', '는', '학교', '에', '간다']

입력: 차를마시며차를탔다
  split  : ['차를마시며차를탔다']
  morphs : ['차', '를', '마시', '며', '차', '를', '탔', '다']



In [3]:
# 📌 §100 조사결합
# ✍️ words 리스트의 각 단어를 split()과 morphs()로 출력해보세요
# 코드를 작성해주세요 👇

words = ["학교를", "학교에", "학교로", "학교가"]
for w in words:
    print(f"  split: {w.split()}  |  morphs: {pecab.morphs(w)}")

# 예상 출력: split: ['학교를']  |  morphs: ['학교', '를'] (4행)

  split: ['학교를']  |  morphs: ['학교', '를']
  split: ['학교에']  |  morphs: ['학교', '에']
  split: ['학교로']  |  morphs: ['학교', '로']
  split: ['학교가']  |  morphs: ['학교', '가']


### 📊 뉴스 제목 — morphs 노이즈 확인

형태소 분석기만으로는 부족하다. 실제 뉴스 제목에 기호·숫자를 넣으면 어떻게 되는지 확인해보자.

In [4]:
# 📌 §100 뉴스 제목 morphs 노이즈
# ✍️ text에 pecab.morphs()를 적용해 어떤 기호·숫자가 토큰으로 잡히는지 확인해보세요
# 코드를 작성해주세요 👇

text = "[속보] 삼성전자, 3분기 영업이익 10조원 돌파"
print(pecab.morphs(text))

# 예상 출력:
# ['[', '속보', ']', '삼성전자', ',', '3', '분기', '영업', '이익', '10', '조원', '돌파']

['[', '속보', ']', '삼성전자', ',', '3', '분기', '영업', '이익', '1', '0', '조', '원', '돌파']


### 🏁 P1-Q1: 비교 표 작성

3문장의 split vs morphs 토큰 수를 표로 정리합니다.

In [5]:
# 📌 §100 비교 표 작성
# split vs morphs 토큰 수 비교 표
print(f"{'문장':<24} {'split':>6} {'morphs':>7}")
print("-" * 40)
for s in sentences:
    n_split  = len(s.split())
    n_morphs = len(pecab.morphs(s))
    print(f"{s:<24} {n_split:>6} {n_morphs:>7}")

# 예상 출력:
# 문장                       split morphs
# 아버지가방에들어가시다          1      7
# 나는학교에간다                  1      5
# 차를마시며차를탔다              1      8

문장                        split  morphs
----------------------------------------
아버지가방에들어가시다                   1       7
나는학교에간다                       1       5
차를마시며차를탔다                     1       8


## 🛠️ P2 — clean_title() 구현

정규식 3단계로 뉴스 제목 노이즈를 제거하는 `clean_title()`을 만든다.

🔗 [Python re 공식 문서](https://docs.python.org/3/library/re.html)

[→ §200 강의노트](../notes/UD-03-200__regex-cheatsheet.md)

### 🔍 re.findall / re.sub — 핵심 함수 확인

`clean_title()`을 만들기 전에, 두 함수의 동작을 먼저 확인한다.

In [6]:
# 📌 §200 re.findall / re.sub 데모
import re
text = "[속보] 삼성전자, 3분기 영업이익 10조원 돌파"

# re.findall: 패턴에 맞는 부분을 전부 찾아 리스트로 반환
print(re.findall(r"\d+", text))
# 출력: ['3', '10']   # 숫자 덩어리를 모두 찾음

# re.sub: 패턴에 맞는 부분을 다른 문자로 교체
print(re.sub(r"\d+", "", text))
# 출력: '[속보] 삼성전자,  분기 영업이익 조원 돌파'

['3', '10']
[속보] 삼성전자, 분기 영업이익 조원 돌파


### 🔬 기본 패턴 10개 — 단독 예시

각 패턴을 단독으로 실행해 동작 원리를 확인한다. `clean_title()` 재료를 먼저 익히자.

In [7]:
# 📌 §200 패턴1 \d — 숫자 한 글자
# ✍️ re.findall(r"\d", "3분기 실적")를 실행하고 결과를 확인해보세요
# 코드를 작성해주세요 👇
import re

text = "3분기 실적"
result = re.findall(r"\d", text)
print(result)

# 예상 출력: ['3']

['3']


In [8]:
# 📌 §200 패턴2 \d+ — 숫자 연속
# ✍️ re.findall(r"\d+", "3분기 영업이익 10조원")를 실행해 \d와 \d+의 차이를 확인해보세요
# 코드를 작성해주세요 👇
import re

text = "3분기 영업이익 10조원"
result = re.findall(r"\d+", text)
print(result)

# 예상 출력: ['3', '10']

['3', '10']


In [9]:
# 📌 §200 패턴3 \s+ — 연속 공백 정리
# ✍️ re.sub(r"\s+", " ", "삼성전자   실적  발표").strip()를 실행해보세요
# 코드를 작성해주세요 👇
import re

text = "삼성전자   실적  발표"
result = re.sub(r"\s+", " ", text).strip()
print(result)

# 예상 출력: '삼성전자 실적 발표'

삼성전자 실적 발표


In [10]:
# 📌 §200 패턴4 ^ — 문자열 시작
# ✍️ re.findall(r"^\[", "[속보] 삼성전자 실적")를 실행해보세요
# 코드를 작성해주세요 👇
import re

text = "[속보] 삼성전자 실적"
result = re.findall(r"^\[", text)
print(result)

# 예상 출력: ['[']

['[']


In [11]:
# 📌 §200 패턴5 $ — 문자열 끝
# ✍️ re.findall(r"다$", "영업이익 발표됐다")를 실행해보세요
# 코드를 작성해주세요 👇
import re

text = "영업이익 발표됐다"
result = re.findall(r"다$", text)
print(result)

# 예상 출력: ['다']

['다']


In [12]:
# 📌 §200 패턴6 [가-힣]+ — 한글 매칭
# ✍️ re.findall(r"[가-힣]+", "삼성전자 Samsung 3분기")를 실행해보세요
# 코드를 작성해주세요 👇
import re

text = "삼성전자 Samsung 3분기"
result = re.findall(r"[가-힣]+", text)
print(result)

# 예상 출력: ['삼성전자', '분기']

['삼성전자', '분기']


In [13]:
# 📌 §200 패턴7 [a-zA-Z]+ — 영문 매칭
# ✍️ re.findall(r"[a-zA-Z]+", "삼성전자 Samsung 3분기")를 실행해보세요
# 코드를 작성해주세요 👇
import re

text = "삼성전자 Samsung 3분기"
result = re.findall(r"[a-zA-Z]+", text)
print(result)

# 예상 출력: ['Samsung']

['Samsung']


In [14]:
# 📌 §200 패턴8 [^가-힣] — 비한글 제거 (부정 집합)
# ✍️ re.sub(r"[^가-힣]", "", "삼성전자, 3분기")를 실행해보세요
# (힌트: [^...] 는 집합에 없는 문자를 전부 제거)
# 코드를 작성해주세요 👇
import re

text = "삼성전자, 3분기"
result = re.sub(r"[^가-힣]", "", text)
print(result)

# 예상 출력: '삼성전자분기'

삼성전자분기


In [15]:
# 📌 §200 패턴9 . 과 * — 와일드카드와 반복
# ✍️ re.findall(r"a.c", "aac abc acc")와 re.findall(r"ab*c", "ac abc abbc abbbc")를 실행해보세요
# 코드를 작성해주세요 👇
import re

# . 예시: 임의의 문자 1개
text = "aac abc acc"
result = re.findall(r"a.c", text)
print(result)
# 출력: ['aac', 'abc', 'acc']
# a.c = a, 임의 문자 1개, c 순서

# * 예시: 0회 이상 반복
text2 = "ac abc abbc abbbc"
result2 = re.findall(r"ab*c", text2)
print(result2)

# 예상 출력:
# ['aac', 'abc', 'acc']
# ['ac', 'abc', 'abbc', 'abbbc']

['aac', 'abc', 'acc']
['ac', 'abc', 'abbc', 'abbbc']


In [16]:
# 📌 §200 패턴10 \[...\] — 이스케이프
# ✍️ re.findall(r"\[[^\]]+\]", "[속보] 삼성 [단독] 카카오")를 실행해보세요
# 코드를 작성해주세요 👇
import re

text = "[속보] 삼성 [단독] 카카오"
result = re.findall(r"\[[^\]]+\]", text)
print(result)

# 예상 출력: ['[속보]', '[단독]']

['[속보]', '[단독]']


In [ ]:
# 📌 §200 Step1 접두 태그 제거
# ✍️ text에서 접두 태그 [속보]를 정규식으로 제거해보세요
# (힌트: re.sub(r"^\[[^\]]+\]\s*", "", text))
# 코드를 작성해주세요 👇

text = "[속보] 삼성전자, 3분기 영업이익 10조원 돌파"

# ^ = 시작, \[ = 리터럴 [, [^\]]+ = ] 아닌 문자들, \] = 리터럴 ], \s* = 뒤 공백
text = re.sub(r"^\[[^\]]+\]\s*", "", text)
print(text)

# 예상 출력: 삼성전자, 3분기 영업이익 10조원 돌파

삼성전자, 3분기 영업이익 10조원 돌파


In [21]:
# 📌 §200 Step2 허용 문자 외 제거
# ✍️ step1 결과에서 한글·영문·공백 외 문자를 공백으로 바꿔보세요
# (힌트: [^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z\s] 패턴 사용)
# 코드를 작성해주세요 👇
text = "[속보] 삼성전자, 3분기 영업이익 10조원 돌파"

# [^...] = ... 에 포함되지 않는 문자 → 공백으로 교체
text = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z\s]", " ", text)
print(text)

# 예상 출력: 삼성전자   분기 영업이익  조원 돌파

 속보  삼성전자   분기 영업이익   조원 돌파


In [22]:
# 📌 §200 Step3 공백 정규화
# ✍️ step2 결과의 연속 공백을 한 칸으로 정리하고 양쪽 공백을 제거해보세요
# (힌트: re.sub + .strip())
# 코드를 작성해주세요 👇
text = re.sub(r"\s+", " ", text).strip()
print(text) 

# 예상 출력: 삼성전자 분기 영업이익 조원 돌파

속보 삼성전자 분기 영업이익 조원 돌파


### 🏁 P2-Q1: clean_title v1 조립

3단계를 하나의 함수로 통합합니다.

In [23]:
# 📌 §200 clean_title v1 통합
# ✍️ 3단계를 하나의 clean_title() 함수로 작성해보세요
# 코드를 작성해주세요 👇

import re

def clean_title(text: str) -> str:
    """v1 — 숫자 제거, 한글·영문·공백만 유지"""
    if not isinstance(text, str):
        return text  # 결측값 그대로 반환
    # 단계 1: 접두 태그 제거
    text = re.sub(r"^\[[^\]]+\]\s*", "", text)
    # 단계 2: 허용 문자 외 제거
    text = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z\s]", " ", text)
    # 단계 3: 공백 정규화
    text = re.sub(r"\s+", " ", text).strip()
    return text

print(clean_title("[속보] 삼성전자, 3분기 영업이익 10조원 돌파"))
# 예상 출력: 삼성전자 분기 영업이익 조원 돌파

삼성전자 분기 영업이익 조원 돌파


In [24]:
# 📌 §200 clean_title v2 숫자 보존
# ✍️ v1을 참고해 숫자(0-9)를 허용 문자에 추가한 clean_title_v2()를 작성해보세요
# (힌트: 2단계 패턴에 0-9 추가)
# 코드를 작성해주세요 👇

def clean_title_v2(text: str) -> str:
    """v2 — 숫자 보존, 0-9 추가"""
    if not isinstance(text, str):
        return text
    text = re.sub(r"^\[[^\]]+\]\s*", "", text)
    text = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z0-9\s]", " ", text)  # 숫자 허용
    text = re.sub(r"\s+", " ", text).strip()
    return text

print(clean_title_v2("[속보] 삼성전자, 3분기 영업이익 10조원 돌파"))
# 예상 출력: 삼성전자 3분기 영업이익 10조원 돌파

삼성전자 3분기 영업이익 10조원 돌파


In [26]:
# 📌 §200 v1 vs v2 비교
# ✍️ 5개 제목에 clean_title v1과 v2를 각각 적용해 결과를 비교 출력해보세요
# 코드를 작성해주세요 👇

titles = [
    "[속보] 삼성전자, 3분기 영업이익 10조원 돌파",
    "[단독] 카카오 CEO 사임…내부 갈등 심화",
    "현대차 전기차 美 수출 30% 급증",
    "네이버·카카오 AI 경쟁 본격화…주가 급등",
    "한국은행 기준금리 3.5%→3.25% 인하 결정",
]

for t in titles:
    print(f"원본: {t}")
    print(f"v1  : {clean_title(t)}")
    print(f"v2  : {clean_title_v2(t)}")
    print()
# 예상 출력: 5쌍 원본/v1/v2 비교

원본: [속보] 삼성전자, 3분기 영업이익 10조원 돌파
v1  : 삼성전자 분기 영업이익 조원 돌파
v2  : 삼성전자 3분기 영업이익 10조원 돌파

원본: [단독] 카카오 CEO 사임…내부 갈등 심화
v1  : 카카오 CEO 사임 내부 갈등 심화
v2  : 카카오 CEO 사임 내부 갈등 심화

원본: 현대차 전기차 美 수출 30% 급증
v1  : 현대차 전기차 수출 급증
v2  : 현대차 전기차 수출 30 급증

원본: 네이버·카카오 AI 경쟁 본격화…주가 급등
v1  : 네이버 카카오 AI 경쟁 본격화 주가 급등
v2  : 네이버 카카오 AI 경쟁 본격화 주가 급등

원본: 한국은행 기준금리 3.5%→3.25% 인하 결정
v1  : 한국은행 기준금리 인하 결정
v2  : 한국은행 기준금리 3 5 3 25 인하 결정



> 💡 **P3 — AI 정규식 생성 + 검증 실습은 Code Drill 파일에 있습니다**
>
> → [UD-03-700 AI 정규식 생성 + 검증](./UD-03-700__ai-regex-student.ipynb)

## 🔍 P4 — Pecab Top30 + Okt 비교

clean_title 통과 제목에서 명사를 추출하고 두 분석기를 비교한다.

🔗 [PeCab GitHub](https://github.com/hyunwoongko/pecab)  
🔗 [KoNLPy 공식 문서](https://konlpy.org/ko/)

[→ §300 강의노트](../notes/UD-03-300__okt-vs-pecab.md)

### 🎯 같은 문장 — 두 분석기 비교

결과부터 먼저 보자. 실행 전에 어느 분석기가 더 세밀하게 분리할지 예측해보세요.

In [ ]:
# 📌 §300 같은 문장 morphs 비교
from konlpy.tag import Okt
from pecab import PeCab

text = "삼성전자 3분기 영업이익 10조원 돌파"
okt = Okt()
pecab = PeCab()
# ✍️ text에 Okt.morphs()와 Pecab.morphs()를 각각 적용해 차이를 확인해보세요
# 코드를 작성해주세요 👇

# 예상 출력:
# Okt.morphs  : ['삼성전자', '3', '분기', '영업이익', '10', '조원', '돌파']
# Pecab.morphs: ['삼성전자', '3', '분기', '영업', '이익', '10', '조원', '돌파']

### 💡 Lego 비유 — pecab.pos() 시연

어절(Lego 세트) → 형태소(블록) → 품사(색깔)

In [ ]:
# 📌 §300 Lego 데모 pecab.pos
# ✍️ pecab.pos("나는학교에간다")를 실행해 각 형태소의 품사 태그를 확인해보세요
# 코드를 작성해주세요 👇

# 예상 출력: [('나', 'NP'), ('는', 'JX'), ('학교', 'NNG'), ('에', 'JKB'), ('가', 'VV'), ('ㄴ다', 'EF')]

### 📊 Stemming 예시 — Okt stem=True

`stem=True`는 어미 변화형을 원형에 가깝게 정규화해 빈도 집계를 안정화한다.

In [ ]:
# 📌 §300 Stemming 예시
# ✍️ okt.morphs("웃겨서 웃겼다 웃으며")를 stem=False, stem=True 각각 실행해보세요
# 코드를 작성해주세요 👇

# 예상 출력:
# stem=False: ['웃겨서', '웃겼다', '웃으며']
# stem=True : ['웃기다', '웃기다', '웃다']

### 🔍 API 비교 — Okt vs Pecab 직접 확인

In [ ]:
# 📌 §300 API 비교 Okt
# ✍️ Okt.morphs(), Okt.pos(norm=True, stem=True), Okt.nouns()를 같은 text에 출력해보세요
# 코드를 작성해주세요 👇

text = "삼성전자 3분기 영업이익 10조원 돌파"
# 예상 출력:
# morphs        : ['삼성전자', '3', '분기', '영업이익', '10', '조원', '돌파']
# nouns         : ['삼성전자', '분기', '영업이익', '조원']

In [ ]:
# 📌 §300 API 비교 Pecab
# ✍️ Pecab.morphs(), Pecab.pos(), Pecab.nouns()를 같은 text에 출력해보세요
# 코드를 작성해주세요 👇

# 예상 출력:
# morphs: ['삼성전자', '3', '분기', '영업', '이익', '10', '조원', '돌파']
# nouns : ['삼성전자', '분기', '영업', '이익', '조원']

### 📊 POS 필터링 — 명사만 추출

품사 태그를 이용해 명사(NNG/NNP)만 추출하는 morphology-aware 방식이다.

In [ ]:
# 📌 §300 POS 필터링 명사 추출
# ✍️ pecab.pos()를 이용해 NNG와 NNP 품사의 단어만 추출해보세요
# (힌트: tag in ("NNG", "NNP") 조건 사용)
# 코드를 작성해주세요 👇

text = "삼성전자 3분기 영업이익 10조원 돌파"
# 예상 출력: ['삼성전자', '분기', '영업', '이익', '조원']

In [ ]:
# 📌 §300 clean_title 적용 준비
# ✍️ titles에 clean_title을 적용해 titles_clean 리스트를 만들고 첫 3개를 출력해보세요
# 코드를 작성해주세요 👇

# 예상 출력:
# 삼성전자 분기 영업이익 조원 돌파
# 카카오 CEO 사임 내부 갈등 심화
# 현대차 전기차 수출 급증

In [ ]:
# 📌 §300 Pecab nouns Top30
from collections import Counter
# ✍️ titles_clean에서 pecab.nouns()를 모아 Counter Top30를 출력해보세요
# (힌트: extend()로 리스트 합치기)
# 코드를 작성해주세요 👇

# 예상 출력: [('카카오', 2), ('AI', 2), ...]

In [ ]:
# 📌 §300 Okt nouns Top30
# ✍️ titles_clean 10개에서 okt.nouns()를 모아 Counter Top30를 출력해보세요
# 코드를 작성해주세요 👇

# 예상 출력: Okt는 복합 명사를 통합하는 경향

### 🏁 P4-Q1: 차이점 관찰 메모

두 분석기의 결과 차이를 집합 연산으로 확인합니다.

In [ ]:
# 📌 §300 차이점 집합 비교
# ✍️ pecab_top30와 okt_top30를 집합으로 변환해 각각만 있는 단어와 공통 단어를 출력해보세요
# 코드를 작성해주세요 👇

# 예상 출력: Pecab only / Okt only / 공통 단어 목록

## 📋 P5 — 전처리 계약 작성

재현 가능한 파이프라인을 위해 전처리 계약서를 작성한다.

[→ §400 강의노트](../notes/UD-03-400__preprocessing-contract.md)

### 💡 Training-Serving Consistency (TSC)

학습에 쓴 전처리 함수를 서비스에도 동일하게 적용해야 한다. 빠뜨리면 어떻게 되는지 확인해보자.

In [ ]:
# 📌 §400 Training-Serving Consistency
# ✍️ preprocess_train (clean_title → nouns)과 preprocess_serve_wrong (nouns만) 함수를 작성하고 차이를 비교해보세요
# 코드를 작성해주세요 👇

t = "[속보] 삼성전자 3분기 실적"
# 예상 출력:
# train  : ['삼성전자', '분기', '실적']
# wrong  : ['속보', '삼성전자', '분기', '실적']

In [ ]:
# 📌 §400 전처리 계약 딕셔너리
# ✍️ 전처리 계약을 딕셔너리로 작성해보세요
# (키: version, input, rules, output, validation)
# 코드를 작성해주세요 👇

preprocessing_contract = {
    # 여기를 채우세요
}
print(preprocessing_contract)
# 예상 출력: 계약 딕셔너리 출력

In [ ]:
# 📌 §400 덮어쓰기 실패
# ✍️ 원본 덮어쓰기가 왜 문제인지 확인해보세요 (나쁜 예 실행 후 원본 복원 시도)
# 코드를 작성해주세요 👇

titles_bad = titles[:]
# 예상 출력: 원본 '[속보]...' 가 정제 결과로 덮어씌워짐

### ⚠️ 불용어 함정 2가지

불용어 제거는 "더 깨끗해지는 것"이 아니다. 잘못 쓰면 분석이 망가진다.

In [ ]:
# 📌 §400 불용어 함정1 의미반전
# ✍️ stopwords에 "안"이 포함된 경우 "이 제품은 안 좋다"의 감성이 어떻게 바뀌는지 확인해보세요
# 코드를 작성해주세요 👇

stopwords = ["안", "못", "의", "이", "가", "은", "는"]
text = "이 제품은 안 좋다"
# 예상 출력:
# 원본 토큰: ['이', '제품', '은', '안', '좋', '다']
# 필터 후  : ['제품', '좋', '다']  ← '안'이 사라져 의미 반전!

In [ ]:
# 📌 §400 불용어 함정2 과삭제
# ✍️ 단음절 불용어 ["이", "가"]를 형태소 리스트에 적용해 결과를 확인해보세요
# 코드를 작성해주세요 👇

stopwords_bad = ["이", "가"]
tokens = ["이빨", "이유", "이메일", "이", "가방", "가"]
# 예상 출력: ['이빨', '이유', '이메일', '가방']  ← 단음절만 제거

### 🏁 P5-Q1: 팀 계약서 완성

계약서에 따라 전체 파이프라인을 실행합니다.

In [ ]:
# 📌 §400 전체 파이프라인 실행
# ✍️ 계약서 기준으로 titles에서 명사를 추출하고 Counter Top10을 출력해보세요
# (힌트: clean_title → pecab.nouns → stopwords 제거 → Counter)
# 코드를 작성해주세요 👇

sw = ["것", "수", "등", "및"]
# 예상 출력: 뉴스 제목 명사 빈도 Top10